When starting this I wanted to read the 3 CSVs in, levereaging Pandas and the flexibility it provides with data frames and ease of manipulation.
The first segment I am combining the 3 CSVs into one. I wanted to start there so I could work off of one CSV, I then realize that I did not need all of them in one for a recommeder, but I kept the one incase I needed to use it in the future, which I did not, but figured I would keep it. I could have removed it out of here, but wanted a bit of my thought process going through this. 
I started by grouping by the movie ID since that is the common on all 3 CSVs. I added tags to the end of it, and set the columns up for combining them all into one. Then I started combining the CSVs, until they were all in one and organized by movie ID. I also organized the columns in a way to make sense to me. 

In [1]:
import pandas as pd

movies = pd.read_csv('C:\\Users\\hessk\\OneDrive\\Desktop\\DSC630\\movies.csv')
tags = pd.read_csv('C:\\Users\\hessk\\OneDrive\\Desktop\\DSC630\\tags.csv')
ratings = pd.read_csv('C:\\Users\\hessk\\OneDrive\\Desktop\\DSC630\\ratings.csv')

print(movies.head(5))
print(tags.head(5))
print(ratings.head(5))

   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  
   userId  movieId              tag   timestamp
0       2    60756            funny  1445714994
1       2    60756  Highly quotable  1445714996
2       2    60756     will ferrell  1445714992
3       2    89774     Boxing story  1445715207
4       2    89774              MMA  1445715200
   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2  

In [3]:
tags_agg = tags.groupby('movieId')['tag'].apply(lambda x: '|'.join(map(str, x))).reset_index()

In [5]:
ratings_movies = ratings.merge(movies, on='movieId', how='left')

In [7]:
combined = ratings_movies.merge(tags_agg, on='movieId', how='left')

In [11]:
combined = combined[['userId', 'movieId', 'title', 'genres', 'rating', 'timestamp', 'tag']]
combined.rename(columns={'tag': 'tags'}, inplace=True)

In [13]:
combined.to_csv('combined_movies_ratings_tags.csv', index=False)

This is where I start the recommender. I used a collaborative filtering recommender system. The reason I chose this one is because I am comparing user to user when finding movies. I find a movie a user likes, then I find another movie that the user rates above a 4, and when a movie is entered, and it matches others, and what they like. My example is toy story, and the print is 10 other movies of people that liked toy story and other movies they also enjoyed as a way to keep the user engaged. It is user to user, so user base collaborative. 
This is where I was trying to get my one big CSV and work on other recommendations based on tags as well, and I kept getting errors, so I learned a lot about recommenders, but could not get my one CSV to work, so I called 2 in, and left tags out. 
Since this is a simple recommeder, I could scale this and add more to it, or add larger data sets. Since it starts with movie ID, finds similar users that gave a similar rating, and pulls the similar ratings and shows the recommended IDs, which is tied to a recommended movie. Then that movie prints at the end of the function. 

In [23]:
def recommend_movies(favorite_movie_title, n=10):
    movie_id = movies[movies['title'].str.lower() == favorite_movie_title.lower()]['movieId'].values
    if len(movie_id) == 0:
        print("Movie not found!")
        return []
    movie_id = movie_id[0]

    similar_users = ratings[(ratings['movieId'] == movie_id) & (ratings['rating'] >= 4)]['userId'].unique()

    similar_ratings = ratings[(ratings['userId'].isin(similar_users)) &
                              (ratings['movieId'] != movie_id) &
                              (ratings['rating'] >= 4)]
    
    rec_counts = similar_ratings['movieId'].value_counts().head(n)
    recommended_ids = rec_counts.index
    
    recommended_movies = movies[movies['movieId'].isin(recommended_ids)][['title', 'genres']]

    return recommended_movies

recommendations = recommend_movies("Toy Story (1995)")
print(recommendations)

                                                  title  \
224           Star Wars: Episode IV - A New Hope (1977)   
257                                 Pulp Fiction (1994)   
277                    Shawshank Redemption, The (1994)   
314                                 Forrest Gump (1994)   
418                                Jurassic Park (1993)   
510                    Silence of the Lambs, The (1991)   
898   Star Wars: Episode V - The Empire Strikes Back...   
900   Raiders of the Lost Ark (Indiana Jones and the...   
911   Star Wars: Episode VI - Return of the Jedi (1983)   
1939                                 Matrix, The (1999)   

                                genres  
224            Action|Adventure|Sci-Fi  
257        Comedy|Crime|Drama|Thriller  
277                        Crime|Drama  
314           Comedy|Drama|Romance|War  
418   Action|Adventure|Sci-Fi|Thriller  
510              Crime|Horror|Thriller  
898            Action|Adventure|Sci-Fi  
900                   